# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row = one content item (content_hash_id), for one client (client_hash_id), aggregated over one calendar month. The raw data ships at daily grain (one row per client + content item + day); I roll this up to monthly by summing volume metrics (impressions, clicks, sessions, etc.) and recalculating gsc_avg_position as a weighted average (sum(gsc_sum_position) / sum(gsc_impressions)) rather than averaging daily averages, since the latter would misweight low-traffic days equally with high-traffic days.

**Time window:** Working month = 2026-03, chosen as a mid-panel month for iteration per the assignment's instructions. The dataset's fact_content_daily_performance_sample file (June 2026, the final available month) is treated as a sealed test month and is never used to develop label or feature logic — only to sanity-check that queries run.

**Verification:** Confirmed the raw daily table has zero duplicate rows per (client, content, date) — 9,841,378 unique combos = 9,841,378 total rows. After monthly aggregation, row count (331,437) exactly matches the count of unique client+content pairs in the raw March data, confirming no rows were lost or duplicated in the rollup. 154,699 rows have zero GSC impressions in March, which correctly produces NaN for gsc_avg_position (undefined average position with no impressions) rather than a computation error.

In [1]:
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

In [2]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "FlyRank/internship-warehouse",
    repo_type="dataset",
    token=hf_token
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"

# Flat files — load directly
df_content = con.sql(f"SELECT * FROM read_parquet('{base}/dim_content.parquet')").df()
df_clients = con.sql(f"SELECT * FROM read_parquet('{base}/dim_clients.parquet')").df()

# March 2026 — go straight to that month's file, no filtering needed
df_march = con.sql(f"SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')").df()

print(f"dim_content: {df_content.shape}")
print(f"dim_clients: {df_clients.shape}")
print(f"March daily rows: {df_march.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dim_content: (519606, 26)
dim_clients: (104, 9)
March daily rows: (9841378, 31)


In [4]:
# Check: does one row in df_march = one client + one content item + one day?
duplicate_check = df_march.groupby(['client_hash_id', 'content_hash_id', 'report_date']).size()
print(f"Any duplicate rows for the same client+content+date? {(duplicate_check > 1).sum()} duplicates found")
print(f"Total unique client+content+date combos: {len(duplicate_check)}")
print(f"Total rows in df_march: {len(df_march)}")

Any duplicate rows for the same client+content+date? 0 duplicates found
Total unique client+content+date combos: 9841378
Total rows in df_march: 9841378


In [5]:
df_march.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

In [7]:
agg_df = df_march.groupby(['client_hash_id', 'content_hash_id']).agg(
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    gsc_sum_position=('gsc_sum_position', 'sum'),
    ga4_pageviews=('ga4_pageviews', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_users=('ga4_users', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
    ga4_total_engagement_sec=('ga4_total_engagement_sec', 'sum'),
    sessions_organic=('sessions_organic', 'sum'),
    sessions_direct=('sessions_direct', 'sum'),
    sessions_referral=('sessions_referral', 'sum'),
    sessions_social=('sessions_social', 'sum'),
    sessions_paid=('sessions_paid', 'sum'),
    sessions_ai=('sessions_ai', 'sum'),
    ai_chatgpt=('ai_chatgpt', 'sum'),
    ai_perplexity=('ai_perplexity', 'sum'),
    ai_gemini=('ai_gemini', 'sum'),
    ai_copilot=('ai_copilot', 'sum'),
    ai_claude=('ai_claude', 'sum'),
    ai_meta=('ai_meta', 'sum'),
    ai_other=('ai_other', 'sum'),
    scroll_events=('scroll_events', 'sum'),
    client_has_gsc=('client_has_gsc', 'max'),
    client_has_ga4=('client_has_ga4', 'max'),
    gsc_data_available=('gsc_data_available', 'max'),
    ga4_data_available=('ga4_data_available', 'max'),
    days_with_data=('report_date', 'nunique'),
).reset_index()

# Recalculate avg position correctly (weighted, not average-of-averages)
agg_df['gsc_avg_position'] = agg_df['gsc_sum_position'] / agg_df['gsc_impressions']

agg_df['month'] = '2026-03'

print(f"Monthly rows: {agg_df.shape}")
agg_df.head()

Monthly rows: (331437, 31)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,...,ai_meta,ai_other,scroll_events,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,days_with_data,gsc_avg_position,month
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,0,0,0,0,0,0,...,0,0,0,True,False,False,<NA>,31,NaN,2026-03
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,0,0,0,0,0,0,...,0,0,0,True,False,False,<NA>,31,NaN,2026-03
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,0,0,0,0,0,0,...,0,0,0,True,False,False,<NA>,31,NaN,2026-03
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9,0,0,0,0,0,...,0,0,0,True,False,True,<NA>,31,9.0,2026-03
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,0,0,0,0,0,0,...,0,0,0,True,False,False,<NA>,31,NaN,2026-03


In [8]:
#checking whether the 0's and NaN's seen mean that they occured due to the division of 0/0
zero_impression_rows = (agg_df['gsc_impressions'] == 0).sum()
nan_position_rows = agg_df['gsc_avg_position'].isna().sum()

print(f"Rows with 0 GSC impressions: {zero_impression_rows}")
print(f"Rows with NaN avg position: {nan_position_rows}")


Rows with 0 GSC impressions: 154699
Rows with NaN avg position: 154699


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context** (needed for joins/filtering, not modeling inputs): client_hash_id, content_hash_id, month, days_with_data, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available

**Feature candidates** (14 — will narrow to 5 in Section 3): gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, scroll_events. All are observed signals known by month-end — no future information, no product-computed scores.

**Label or proxy:** Intentionally empty this week. A decline/opportunity label requires comparing at least two months (e.g. prior 90 days → next 30 days per the lane guide), but this contract currently works within a single month (March 2026). Building a real proxy label is out of scope until a second time window is added — documented as a limitation in Section 4.

**Excluded:**

gsc_sum_position — raw intermediate value used only to calculate

gsc_avg_position; not meaningful as a standalone feature on its own.

sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — AI-referral signals are excluded this week. Per the lane guide, only 30,177 of 78.8M daily rows have any AI session data — too sparse to use safely as features without producing misleading results.

In [10]:
field_buckets = {
    'context': [
        'client_hash_id', 'content_hash_id', 'month', 'days_with_data',
        'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available'
    ],
    'feature': [
        'gsc_impressions', 'gsc_clicks', 'gsc_avg_position',
        'ga4_pageviews', 'ga4_sessions', 'ga4_users',
        'ga4_engaged_sessions', 'ga4_total_engagement_sec',
        'sessions_organic', 'sessions_direct', 'sessions_referral',
        'sessions_social', 'sessions_paid', 'scroll_events'
    ],
    'excluded': [
        'gsc_sum_position',  # raw intermediate used only to compute gsc_avg_position — not a standalone feature
        'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini',
        'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other'
        # AI-referral columns — lane guide flags these as too sparse
        # (30,177 rows with AI sessions out of 78.8M total) to use safely this week
    ],
    'label_or_proxy': []  # intentionally empty — no valid label possible from a single month; addressed in Section 4
}

# sanity check: did we account for every column?
all_bucketed = sum(field_buckets.values(), [])
missing = set(agg_df.columns) - set(all_bucketed)
print(f"Columns not yet bucketed: {missing}")

Columns not yet bucketed: set()


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.